# TASK 3 — Synthetic AI medical trust survey analysis
Sharipkhan Merey & Olzhabek Marzhan. **Synthetic smoke-test only**; no real participants or hypothesis confirmation. The Task 2 CSV has no cold-start timing runs, so warm-up filtering is not applicable.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from process_data import analyze
from visualize import generate
np.random.seed(42)

## Data ingestion and quality guardrails
Duplicate IDs, invalid groups, and trust scores outside 1–5 are excluded. No warm-up phase exists in these survey data.

In [2]:
clean, summary, quality = analyze(ROOT / "data/sample/sample_synthetic.csv", ROOT / "data/processed/metrics_summary.csv")
print(quality)
display(clean.head())

{'total_rows': 10, 'valid_rows': 10, 'invalid_rows': 0, 'completion_rate': 1.0, 'invalid_response_rate': 0.0}


,response_id,participant_group,trust_score,prior_ai_use,ai_knowledge,perceived_accuracy,explanation_clarity,doctor_supervision,age,education
0,S001,student,4.0,yes,4,4,4,5,20,undergraduate
1,S002,student,3.0,yes,3,3,4,4,21,undergraduate
2,S003,student,5.0,yes,5,5,5,5,19,undergraduate
3,S004,student,2.0,no,2,2,2,4,22,undergraduate
4,S005,student,4.0,yes,4,4,3,5,23,undergraduate


## Statistical aggregation
Median, p25/p75, p95/p99, sample standard deviation and percentile-bootstrap 95% CI for the group mean. With n=5 per group these intervals are illustrative only.

In [3]:
display(summary.round(3))
means = summary.set_index("group")["mean"]
print("Student minus patient mean trust:", round(float(means["student"]-means["patient"]),3))

,group,n,mean,sd,median,p25,p75,p95,p99,ci95_low,ci95_high,invalid_rate
0,patient,5,2.8,0.837,3.0,2.0,3.0,3.8,3.96,2.2,3.405,0.0
1,student,5,3.6,1.140,4.0,3.0,4.0,4.8,4.96,2.6,4.400,0.0


Student minus patient mean trust: 0.8


## Visualization and export
Mean trust with 95% bootstrap confidence intervals and a distribution boxplot. Figures exported as PDF and 300-DPI PNG.

In [4]:
generate(clean,summary,ROOT / "reports/figures")
print(sorted(x.name for x in (ROOT / "reports/figures").iterdir()))

['fig1_mean_trust_ci.pdf', 'fig1_mean_trust_ci.png', 'fig2_trust_distribution.pdf', 'fig2_trust_distribution.png']


## Limitations
These are ten synthetic responses (five per group), so no real-world statistical or medical conclusions can be drawn. The assignment’s latency/load example is not a valid metric for this survey; plots use trust score (Likert points). No real experimental warm-up logs or measured error rates exist; invalid-response rate is the appropriate survey guardrail.